In [1]:
import csv
import hashlib
import io
import platform
import shutil
import subprocess
from collections import Counter, defaultdict
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path, PurePosixPath


def mounted_dataset(slug):
    for root in (Path("/kaggle/input/datasets/nayebul0"), Path("/kaggle/input")):
        candidate = root / slug
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(f"Attach this Kaggle input: nayebul0/{slug}")


print("Python:", platform.python_version())
for package in ("tensorflow", "keras", "numpy", "pillow"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: distribution not found")
print(f"Free working space: {shutil.disk_usage('/kaggle/working').free / 2**30:.1f} GiB")

gpu_tool = shutil.which("nvidia-smi")
if gpu_tool:
    gpu = subprocess.run(
        [gpu_tool, "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True
    )
    print("GPU, VRAM, driver:", gpu.stdout.strip() or gpu.stderr.strip() or "No output")
else:
    print("GPU: nvidia-smi unavailable")

d2_image_root = mounted_dataset("assignment02-d2-rice-leaf-5932") / "Rice Leaf Disease Images"
d2_manifest_path = mounted_dataset("assignment02-verified-train-val-test-splits") / "D2_split_manifest.csv"
assert d2_image_root.is_dir(), f"D2 image root missing: {d2_image_root}"

raw = d2_manifest_path.read_bytes()
manifest_hash = hashlib.sha256(raw).hexdigest()
print("D2 image root:", d2_image_root)
print("D2 manifest SHA-256:", manifest_hash)
assert manifest_hash == "261b97b6fa700197f2dde6bc0f833bf802bd1301c0611606400e990cfad2a185"

d2_class_map = {0: "Bacterial Blight", 1: "Blast", 2: "Brown Spot", 3: "Tungro"}
rows = list(csv.DictReader(io.StringIO(raw.decode("utf-8-sig"))))
counts = Counter(row["split"] for row in rows)
assert counts == {"train": 3327, "val": 712, "test": 713}, counts

groups = defaultdict(set)
classes_by_split = defaultdict(set)
seen_paths = set()
file_issues = []
for row in rows:
    relative = PurePosixPath(row["rel_path"])
    class_id = int(row["class_id"])
    assert row["dataset"] == "D2" and d2_class_map.get(class_id) == row["class_name"], row
    assert relative.parts and not relative.is_absolute() and ".." not in relative.parts, row
    assert relative not in seen_paths and row["leakage_group_id"], row
    seen_paths.add(relative)
    groups[row["leakage_group_id"]].add(row["split"])
    classes_by_split[row["split"]].add(class_id)

    image_path = d2_image_root.joinpath(*relative.parts)
    if not image_path.is_file():
        file_issues.append(("missing", relative.as_posix()))
        continue

    with image_path.open("rb") as stream:
        digest = hashlib.sha256()
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)

    if digest.hexdigest() != row["file_sha256"]:
        file_issues.append(("hash mismatch", relative.as_posix()))

assert all(classes_by_split[split] == set(d2_class_map) for split in counts)
crossing_groups = sum(len(splits) > 1 for splits in groups.values())
print("D2 train/val/test:", tuple(counts[split] for split in ("train", "val", "test")))
print("D2 classes:", d2_class_map)
print("D2 crossing groups:", crossing_groups)
print("D2 file issues:", len(file_issues))
assert crossing_groups == 0 and not file_issues, (crossing_groups, file_issues[:5])
print(f"D2 PREFLIGHT PASSED: {len(rows):,} verified image files")


Python: 3.13.16
tensorflow: 2.21.0
keras: 3.13.2
numpy: 2.1.3
pillow: 13.0.0
Free working space: 19.5 GiB
GPU, VRAM, driver: Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04
D2 image root: /kaggle/input/datasets/nayebul0/assignment02-d2-rice-leaf-5932/Rice Leaf Disease Images
D2 manifest SHA-256: 261b97b6fa700197f2dde6bc0f833bf802bd1301c0611606400e990cfad2a185
D2 train/val/test: (3327, 712, 713)
D2 classes: {0: 'Bacterial Blight', 1: 'Blast', 2: 'Brown Spot', 3: 'Tungro'}
D2 crossing groups: 0
D2 file issues: 0
D2 PREFLIGHT PASSED: 4,752 verified image files


In [2]:
import csv
import hashlib
import io
import os
import random
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

import cv2
import numpy as np
from IPython.display import FileLink, display
from PIL import Image, ImageDraw, ImageFont, ImageOps

seed = 2026
size = 224
working = Path("/kaggle/working")
expected_hash = "261b97b6fa700197f2dde6bc0f833bf802bd1301c0611606400e990cfad2a185"


def mounted_dataset(slug):
    for parent in (Path("/kaggle/input/datasets/nayebul0"), Path("/kaggle/input")):
        candidate = parent / slug
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(f"Attach this Kaggle input: nayebul0/{slug}")


def resize_to_fit(rgb):
    height, width = rgb.shape[:2]
    scale = size / max(height, width)
    dimensions = (max(1, round(width * scale)), max(1, round(height * scale)))
    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    return cv2.resize(rgb, dimensions, interpolation=interpolation)


def stretch_luminance(rgb):
    lab = cv2.cvtColor(rgb.astype(np.float32) / 255.0, cv2.COLOR_RGB2LAB)
    lightness = lab[:, :, 0]
    low = float(lightness.min())
    high = float(lightness.max())
    if high == low:
        return rgb.copy()
    lab[:, :, 0] = (lightness - low) * (100.0 / (high - low))
    stretched = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    return np.rint(np.clip(stretched, 0.0, 1.0) * 255.0).astype(np.uint8)


def bright_otsu_mask(gray):
    histogram = np.bincount(gray.ravel(), minlength=256).astype(np.int64)
    count_below = np.cumsum(histogram)[:-1]
    sum_below = np.cumsum(histogram * np.arange(256, dtype=np.int64))[:-1]
    count_above = gray.size - count_below
    valid = (count_below > 0) & (count_above > 0)
    if not valid.any():
        return np.ones(gray.shape, dtype=bool), None

    total_intensity = int(histogram @ np.arange(256, dtype=np.int64))
    separation = gray.size * sum_below[valid] - total_intensity * count_below[valid]
    scores = separation.astype(np.float64) ** 2 / (count_below[valid] * count_above[valid])
    threshold = int(np.flatnonzero(valid)[np.argmax(scores)] + 1)
    return gray >= threshold, threshold


def padded_panel(array):
    image = Image.fromarray(array).convert("RGB")
    canvas = Image.new("RGB", (size, size))
    canvas.paste(image, ((size - image.width) // 2, (size - image.height) // 2))
    return canvas.resize((160, 160), Image.Resampling.NEAREST)


def select_audit_rows(rows):
    rng = random.Random(seed)
    selected = {}
    for class_id in range(4):
        selected[class_id] = []
        for split, count in (("train", 8), ("val", 4)):
            eligible = sorted(
                (row for row in rows if row["split"] == split and int(row["class_id"]) == class_id),
                key=lambda row: row["rel_path"]
            )
            selected[class_id].extend(rng.sample(eligible, count))
    return selected


def run_d2_audit(image_root, manifest_path, output_dir):
    raw = manifest_path.read_bytes()
    assert hashlib.sha256(raw).hexdigest() == expected_hash, "D2 frozen manifest hash mismatch"
    rows = list(csv.DictReader(io.StringIO(raw.decode("utf-8-sig"))))
    selected = select_audit_rows(rows)
    assert sum(map(len, selected.values())) == 48
    assert {row["split"] for group in selected.values() for row in group} == {"train", "val"}

    report = []
    font = ImageFont.load_default(size=14)
    title_font = ImageFont.load_default(size=18)
    columns = ("Resized RGB", "Stretched L*", "Otsu mask", "Masked RGB")
    archive_path = output_dir / "D2_Step_2_Otsu_Audit.zip"
    print(f"OpenCV {cv2.__version__} | seed {seed} | bright pixels retained | 32 train + 16 val")

    with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
        for class_id, class_rows in selected.items():
            class_name = class_rows[0]["class_name"]
            sheet = Image.new("RGB", (684, 68 + 190 * len(class_rows)), "#f4f4f4")
            draw = ImageDraw.Draw(sheet)
            draw.text((10, 10), f"D2 · {class_name} · class {class_id}", font=title_font, fill="black")
            for col, label in enumerate(columns):
                draw.text((10 + 168 * col, 43), label, font=font, fill="black")

            class_report = []
            for index, row in enumerate(class_rows, start=1):
                with Image.open(image_root / row["rel_path"]) as source:
                    rgb = np.asarray(ImageOps.exif_transpose(source).convert("RGB"))

                resized = resize_to_fit(rgb)
                stretched = stretch_luminance(resized)
                gray = cv2.cvtColor(stretched, cv2.COLOR_RGB2GRAY)
                mask, threshold = bright_otsu_mask(gray)
                masked = np.where(mask[:, :, None], stretched, np.uint8(0))
                fraction = float(mask.mean())
                flag = threshold is None or fraction <= 0.02 or fraction >= 0.98
                record = {
                    "class_id": class_id,
                    "class_name": class_name,
                    "sheet_row": index,
                    "split": row["split"],
                    "rel_path": row["rel_path"],
                    "otsu_threshold": threshold,
                    "foreground_fraction": f"{fraction:.6f}",
                    "constant_gray": threshold is None,
                    "coverage_flag": flag
                }
                class_report.append(record)
                top = 68 + 190 * (index - 1)
                panels = (resized, stretched, mask.astype(np.uint8) * 255, masked)
                for col, panel in enumerate(panels):
                    sheet.paste(padded_panel(panel), (10 + 168 * col, top))
                threshold_text = "constant" if threshold is None else str(threshold)
                label = f"Row {index:02d}  {row['split']}  |  threshold {threshold_text}  |  foreground {fraction:.1%}"
                if flag:
                    label += "  |  FLAG"
                draw.text((10, top + 163), label, font=font, fill="black")

            png = io.BytesIO()
            sheet.save(png, format="PNG")
            archive.writestr(f"D2_class_{class_id:02d}.png", png.getvalue())
            report.extend(class_report)
            coverage = np.array([float(row["foreground_fraction"]) for row in class_report])
            flagged = sum(row["coverage_flag"] for row in class_report)
            print(
                f"{class_id} {class_name}: "
                f"foreground min/median/max {coverage.min():.1%}/{np.median(coverage):.1%}/{coverage.max():.1%}"
                f" | flags {flagged}/12"
            )

        csv_text = io.StringIO(newline="")
        writer = csv.DictWriter(csv_text, fieldnames=report[0].keys())
        writer.writeheader()
        writer.writerows(report)
        archive.writestr("D2_otsu_audit.csv", csv_text.getvalue())

    with ZipFile(archive_path) as archive:
        assert len(archive.namelist()) == 5 and archive.testzip() is None

    print(f"D2 OTSU AUDIT EXPORTED: {len(report)} images, {len(selected)} class sheets")
    print(f"Audit archive: {archive_path}")
    display(FileLink(os.path.relpath(archive_path)))


image_root = mounted_dataset("assignment02-d2-rice-leaf-5932") / "Rice Leaf Disease Images"
manifest = mounted_dataset("assignment02-verified-train-val-test-splits") / "D2_split_manifest.csv"
run_d2_audit(image_root, manifest, working)


OpenCV 4.14.0 | seed 2026 | bright pixels retained | 32 train + 16 val
0 Bacterial Blight: foreground min/median/max 13.8%/45.3%/69.2% | flags 0/12
1 Blast: foreground min/median/max 18.2%/39.5%/63.0% | flags 0/12
2 Brown Spot: foreground min/median/max 20.8%/47.4%/87.1% | flags 0/12
3 Tungro: foreground min/median/max 18.0%/52.8%/59.3% | flags 0/12
D2 OTSU AUDIT EXPORTED: 48 images, 4 class sheets
Audit archive: /kaggle/working/D2_Step_2_Otsu_Audit.zip


/kaggle/working/D2_Step_2_Otsu_Audit.zip

In [3]:
from math import prod

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

seed = 2026
image_size = 224
num_classes = 4


def build_vgg_path(inputs):
    x = inputs
    for block_id, widths in enumerate(((64,), (128,), (256, 256), (512, 512)), start=1):
        for conv_id, width in enumerate(widths, start=1):
            x = layers.Conv2D(
                width, 3, padding="same", activation="relu",
                name=f"block{block_id}_conv{conv_id}"
            )(x)
        x = layers.MaxPooling2D(2, strides=2, name=f"block{block_id}_pool")(x)
    x = layers.Flatten(name="vgg_flatten")(x)
    return layers.Dense(
        512, activation="relu", kernel_initializer="he_normal", name="vgg_projection"
    )(x)


def residual_bottleneck(inputs, width, name, projection):
    shortcut = inputs
    if projection:
        shortcut = layers.Conv2D(4 * width, 1, name=f"{name}_0_conv")(shortcut)
        shortcut = layers.BatchNormalization(epsilon=1.001e-5, name=f"{name}_0_bn")(shortcut)

    x = inputs
    for conv_id, (filters, kernel_size) in enumerate(((width, 1), (width, 3), (4 * width, 1)), start=1):
        x = layers.Conv2D(
            filters, kernel_size, strides=1, padding="same", name=f"{name}_{conv_id}_conv"
        )(x)
        x = layers.BatchNormalization(epsilon=1.001e-5, name=f"{name}_{conv_id}_bn")(x)
        if conv_id < 3:
            x = layers.Activation("relu", name=f"{name}_{conv_id}_relu")(x)

    x = layers.Add(name=f"{name}_add")([shortcut, x])
    return layers.Activation("relu", name=f"{name}_out")(x)


def build_resnet_path(inputs):
    x = layers.ZeroPadding2D(3, name="conv1_pad")(inputs)
    x = layers.Conv2D(64, 7, strides=2, name="conv1_conv")(x)
    x = layers.BatchNormalization(epsilon=1.001e-5, name="conv1_bn")(x)
    x = layers.Activation("relu", name="conv1_relu")(x)
    x = layers.ZeroPadding2D(1, name="pool1_pad")(x)
    x = layers.MaxPooling2D(3, strides=2, name="pool1_pool")(x)

    blocks = (
        ("conv2_block1", 64, True, True),
        ("conv2_block2", 64, False, False),
        ("conv3_block1", 128, True, True),
        ("conv3_block2", 128, False, False),
        ("conv4_block1", 256, True, False)
    )
    for name, width, projection, downsample in blocks:
        x = residual_bottleneck(x, width, name, projection)
        if downsample:
            x = layers.MaxPooling2D(3, strides=2, padding="same", name=f"{name}_pool")(x)

    return layers.GlobalAveragePooling2D(name="resnet_gap")(x)


def self_gate(features, name):
    weights = layers.Activation("sigmoid", name=f"{name}_sigmoid")(features)
    return layers.Multiply(name=f"{name}_gated")([features, weights])


def build_amff(class_count):
    inputs = keras.Input(shape=(image_size, image_size, 3), dtype="float32", name="prepared_image")
    vgg = build_vgg_path(inputs)
    resnet = build_resnet_path(inputs)
    fusion = layers.Concatenate(name="amff_fusion")([
        self_gate(vgg, "vgg"),
        self_gate(resnet, "resnet")
    ])
    x = layers.Dense(
        512, activation="relu", kernel_initializer="he_normal", name="fusion_dense"
    )(fusion)
    x = layers.Dropout(0.5, name="fusion_dropout")(x)
    outputs = layers.Dense(class_count, activation="softmax", name="class_probabilities")(x)
    return keras.Model(inputs, outputs, name="amff_hybrid_d2")


def verify_d2_graph(model):
    expected_shapes = {
        "block4_pool": (14, 14, 512),
        "vgg_flatten": (100352,),
        "vgg_projection": (512,),
        "pool1_pool": (56, 56, 64),
        "conv2_block1_pool": (28, 28, 256),
        "conv3_block1_pool": (14, 14, 512),
        "conv4_block1_out": (14, 14, 1024),
        "resnet_gap": (1024,),
        "amff_fusion": (1536,),
        "class_probabilities": (4,)
    }
    for name, expected in expected_shapes.items():
        actual = tuple(model.get_layer(name).output.shape[1:])
        assert actual == expected, (name, actual, expected)
        print(f"{name}: {actual}")

    gate_names = ("vgg_sigmoid", "vgg_gated", "resnet_sigmoid", "resnet_gated")
    gate_params = sum(model.get_layer(name).count_params() for name in gate_names)
    trainable = sum(prod(weight.shape) for weight in model.trainable_weights)
    nontrainable = sum(prod(weight.shape) for weight in model.non_trainable_weights)
    total = model.count_params()
    print(f"Parameters: {trainable:,} trainable | {nontrainable:,} nontrainable | {total:,} total")
    print(f"Gate parameters: {gate_params}")
    assert (gate_params, trainable, nontrainable, total) == (0, 59003140, 11392, 59014532)

    probe = keras.Model(model.input, [
        model.get_layer(name).output for name in
        ("vgg_projection", "resnet_gap", "vgg_gated", "resnet_gated", "amff_fusion", "class_probabilities")
    ])
    sample = tf.random.stateless_uniform(
        (1, image_size, image_size, 3), seed=(seed, 0), minval=-128.0, maxval=128.0
    )
    vgg, resnet, gated_vgg, gated_resnet, fused, probs = probe(sample, training=False)
    tf.debugging.assert_near(gated_vgg, vgg * tf.math.sigmoid(vgg))
    tf.debugging.assert_near(gated_resnet, resnet * tf.math.sigmoid(resnet))
    tf.debugging.assert_near(fused, tf.concat([gated_vgg, gated_resnet], axis=-1))
    tf.debugging.assert_all_finite(probs, "Nonfinite class probabilities")
    tf.debugging.assert_near(tf.reduce_sum(probs, axis=-1), tf.ones((1,)))
    print(f"Synthetic forward pass: {tuple(probs.shape)} | probability sum {float(tf.reduce_sum(probs)):.6f}")


keras.mixed_precision.set_global_policy("float32")
keras.utils.set_random_seed(seed)
print("TensorFlow:", tf.__version__)
print("TensorFlow-visible GPUs:", tf.config.list_physical_devices("GPU"))
d2_amff = build_amff(num_classes)
verify_d2_graph(d2_amff)
print("D2 AMFF ARCHITECTURE PASSED")


TensorFlow: 2.21.0
TensorFlow-visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
block4_pool: (14, 14, 512)
vgg_flatten: (100352,)
vgg_projection: (512,)
pool1_pool: (56, 56, 64)
conv2_block1_pool: (28, 28, 256)
conv3_block1_pool: (14, 14, 512)
conv4_block1_out: (14, 14, 1024)
resnet_gap: (1024,)
amff_fusion: (1536,)
class_probabilities: (4,)
Parameters: 59,003,140 trainable | 11,392 nontrainable | 59,014,532 total
Gate parameters: 0
Synthetic forward pass: (1, 4) | probability sum 1.000000
D2 AMFF ARCHITECTURE PASSED


In [4]:
import hashlib
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras

init_dir = Path("/kaggle/working/paper5_d2_initialization")
init_dir.mkdir(parents=True, exist_ok=True)
init_seed = 2026

vgg_names = (
    "block1_conv1", "block2_conv1", "block3_conv1",
    "block3_conv2", "block4_conv1", "block4_conv2"
)
resnet_names = ["conv1_conv", "conv1_bn"]
for block, projection in (
    ("conv2_block1", True), ("conv2_block2", False),
    ("conv3_block1", True), ("conv3_block2", False), ("conv4_block1", True)
):
    for index in range(0 if projection else 1, 4):
        resnet_names.extend((f"{block}_{index}_conv", f"{block}_{index}_bn"))

donors = (
    (
        "VGG16", keras.applications.VGG16,
        "https://storage.googleapis.com/tensorflow/keras-applications/vgg16/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5",
        "6d6bbae143d832006294945121d1f1fc", vgg_names, (6, 12, 4500864)
    ),
    (
        "ResNet50", keras.applications.ResNet50,
        "https://storage.googleapis.com/tensorflow/keras-applications/resnet/resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5",
        "4d473c1dd8becc155b73f8504c6f6626", resnet_names, (38, 114, 2343936)
    )
)


def digest_file(path, algorithm):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, algorithm).hexdigest()


def donor_file(url, expected_md5):
    filename = url.rsplit("/", 1)[-1]
    for candidate in sorted(Path("/kaggle/input").rglob(filename)):
        if candidate.is_file() and digest_file(candidate, "md5") == expected_md5:
            return candidate

    path = Path(keras.utils.get_file(
        fname=filename, origin=url, file_hash=expected_md5, hash_algorithm="md5",
        cache_dir=str(init_dir), cache_subdir="imagenet"
    ))
    assert digest_file(path, "md5") == expected_md5, path
    return path


def layer_weight_digest(model, layer_name):
    digest = hashlib.sha256()
    for variable in model.get_layer(layer_name).weights:
        digest.update(variable.numpy())
    return digest.hexdigest()


def transfer_layers(target, constructor, path, layer_names):
    transferred = []
    with tf.device("/CPU:0"):
        source = constructor(include_top=False, weights=None, input_shape=(224, 224, 3))
        source.load_weights(str(path))
        for name in layer_names:
            source_arrays = source.get_layer(name).get_weights()
            target_layer = target.get_layer(name)
            target_shapes = [tuple(variable.shape) for variable in target_layer.weights]
            source_shapes = [array.shape for array in source_arrays]
            assert source_shapes and source_shapes == target_shapes, (name, source_shapes, target_shapes)
            assert all(np.isfinite(array).all() for array in source_arrays), name

            target_layer.set_weights(source_arrays)
            for original, copied in zip(source_arrays, target_layer.get_weights(), strict=True):
                np.testing.assert_array_equal(original, copied, err_msg=name)

            transferred.append({
                "source_layer": name,
                "target_layer": name,
                "tensor_shapes": source_shapes,
                "scalars": sum(array.size for array in source_arrays)
            })
    return transferred


keras.utils.set_random_seed(init_seed)
d2_amff = build_amff(4)
assert d2_amff.count_params() == 59014532
dense_names = ("vgg_projection", "fusion_dense", "class_probabilities")
expected_weighted_layers = set(vgg_names) | set(resnet_names) | set(dense_names)
assert {layer.name for layer in d2_amff.layers if layer.weights} == expected_weighted_layers
dense_digests = {name: layer_weight_digest(d2_amff, name) for name in dense_names}

donor_records = []
for name, constructor, url, expected_md5, layer_names, expected_counts in donors:
    path = donor_file(url, expected_md5)
    sha256 = digest_file(path, "sha256")
    print(f"{name} file: {path.name}")
    print(f"{name} SHA-256: {sha256}")
    mappings = transfer_layers(d2_amff, constructor, path, layer_names)
    counts = (
        len(mappings),
        sum(len(mapping["tensor_shapes"]) for mapping in mappings),
        sum(mapping["scalars"] for mapping in mappings)
    )
    assert counts == expected_counts, (name, counts, expected_counts)
    print(f"{name} verified: {counts[0]} layers | {counts[1]} tensors | {counts[2]:,} scalars")
    donor_records.append({
        "application": name, "source_url": url, "filename": path.name,
        "md5": expected_md5, "sha256": sha256, "mappings": mappings
    })

assert all(layer_weight_digest(d2_amff, name) == value for name, value in dense_digests.items())
print("All three seeded dense layers verified.")

probe = tf.random.stateless_uniform((1, 224, 224, 3), seed=(init_seed, 1), minval=-128.0, maxval=128.0)
probabilities = d2_amff(probe, training=False)
tf.debugging.assert_all_finite(probabilities, "Nonfinite class probabilities after transfer")
tf.debugging.assert_near(tf.reduce_sum(probabilities, axis=-1), tf.ones((1,)))

checkpoint_path = init_dir / "D2_amff_shared_init.weights.h5"
temporary_checkpoint = init_dir / "D2_amff_shared_init.tmp.weights.h5"
d2_amff.save_weights(str(temporary_checkpoint))
temporary_checkpoint.replace(checkpoint_path)
checkpoint_sha256 = digest_file(checkpoint_path, "sha256")

report = {
    "dataset": "D2", "seed": init_seed, "tensorflow": tf.__version__,
    "keras": keras.__version__, "checkpoint": checkpoint_path.name,
    "checkpoint_sha256": checkpoint_sha256, "seeded_dense_layer_sha256": dense_digests,
    "donors": donor_records
}
report_path = init_dir / "D2_imagenet_transfer.json"
report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(f"Shared initialization: {checkpoint_path}")
print(f"Shared initialization SHA-256: {checkpoint_sha256}")
print(f"Weight transfer report: {report_path}")
print("D2 IMAGENET INITIALIZATION PASSED")


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
VGG16 file: vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5
VGG16 SHA-256: bfe5187d0a272bed55ba430631598124cff8e880b98d38c9e56c8d66032abdc1
VGG16 verified: 6 layers | 12 tensors | 4,500,864 scalars
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
ResNet50 file: resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5
ResNet50 SHA-256: 66c8b43daff3fcc15bc4f30e3d2a167e21a14d9c9598a5394e5516471f4af504
ResNet50 verified: 38 layers | 114 tensors | 2,343,936 scalars
All three seeded dense layers verified.
Shared initialization: /kaggle/working/paper5_d2_initialization/D2_amff_shared_init.weights.h5
Shared initialization SHA-256: 545cade231e29e4c552ef469e0d0205d567fb2efa4b82e5fe720715f9a9e0506
Weight transfer report: /kaggle/working/paper5_d2_initialization/D2_imagenet_transfer.json
D2 IMAGENET INITIALIZATION PASSED


In [5]:
import csv
import hashlib
import io
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageOps
from tensorflow import keras

d2_data_seed = 2026
d2_size = 224
d2_batch_size = 30
d2_cache_dir = Path("/kaggle/working/paper5_d2_preprocessed")
d2_modes = ("otsu", "stretch_only")
d2_fit_counts = {"train": 3327, "val": 712}


def read_d2_manifest(manifest_path):
    raw = manifest_path.read_bytes()
    expected = "261b97b6fa700197f2dde6bc0f833bf802bd1301c0611606400e990cfad2a185"
    assert hashlib.sha256(raw).hexdigest() == expected, "D2 frozen manifest hash mismatch"
    return list(csv.DictReader(io.StringIO(raw.decode("utf-8-sig"))))


def center_pad_d2(rgb):
    height, width = rgb.shape[:2]
    top = (d2_size - height) // 2
    left = (d2_size - width) // 2
    canvas = np.zeros((d2_size, d2_size, 3), dtype=np.uint8)
    canvas[top:top + height, left:left + width] = rgb
    return canvas


def prepare_d2_image(path):
    with Image.open(path) as source:
        rgb = np.asarray(ImageOps.exif_transpose(source).convert("RGB"))
    stretched = stretch_luminance(resize_to_fit(rgb))
    gray = cv2.cvtColor(stretched, cv2.COLOR_RGB2GRAY)
    mask, _ = bright_otsu_mask(gray)
    masked = np.where(mask[:, :, None], stretched, np.uint8(0))
    return center_pad_d2(masked), center_pad_d2(stretched)


def build_d2_fit_cache(image_root, fit_rows):
    d2_cache_dir.mkdir(parents=True, exist_ok=True)
    shape = (len(fit_rows), d2_size, d2_size, 3)
    pending = {mode: d2_cache_dir / f"{mode}.tmp.npy" for mode in d2_modes}
    arrays = {
        mode: np.lib.format.open_memmap(path, mode="w+", dtype=np.uint8, shape=shape)
        for mode, path in pending.items()
    }
    for index, row in enumerate(fit_rows):
        masked, stretched = prepare_d2_image(image_root / row["rel_path"])
        arrays["otsu"][index] = masked
        arrays["stretch_only"][index] = stretched

    for mode in d2_modes:
        arrays[mode].flush()
        del arrays[mode]
        pending[mode].replace(d2_cache_dir / f"{mode}.npy")

    with (d2_cache_dir / "fit_cache_index.csv").open("w", newline="", encoding="utf-8") as stream:
        fields = ("cache_index", "rel_path", "class_id", "class_name", "split")
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        for index, row in enumerate(fit_rows):
            writer.writerow({"cache_index": index, **{key: row[key] for key in fields[1:]}})


def d2_affine_matrix(epoch, row_index):
    rng = np.random.default_rng([d2_data_seed, epoch, int(row_index)])
    offset = rng.uniform(-25.0, 25.0, size=2)
    scales = rng.uniform(0.9, 1.1, size=2)
    angle = float(rng.uniform(-20.0, 20.0))
    center = ((d2_size - 1) / 2,) * 2
    rotation = cv2.getRotationMatrix2D(center, angle, 1.0)
    linear = rotation[:, :2] @ np.diag(scales)
    translation = np.asarray(center) + offset - linear @ center
    return np.column_stack((linear, translation))


def d2_imagenet_input(rgb_batch):
    batch = rgb_batch.astype(np.float32)
    return np.ascontiguousarray(keras.applications.vgg16.preprocess_input(batch))


class D2FitBatches(keras.utils.PyDataset):
    def __init__(self, images, labels, row_indices, training, batch_size):
        super().__init__(workers=1, use_multiprocessing=False)
        self.images = images
        self.labels = labels
        self.row_indices = row_indices
        self.training = training
        self.batch_size = batch_size
        self.epoch = 0
        self.order = self.epoch_order()

    def epoch_order(self):
        if self.training:
            return np.random.default_rng([d2_data_seed, self.epoch]).permutation(self.row_indices)
        return self.row_indices

    def __len__(self):
        return (len(self.order) + self.batch_size - 1) // self.batch_size

    def batch_rows(self, index):
        start = index * self.batch_size
        return self.order[start:start + self.batch_size]

    def __getitem__(self, index):
        indices = self.batch_rows(index)
        rgb_batch = self.images[indices]
        if self.training:
            for position, row_index in enumerate(indices):
                rgb_batch[position] = cv2.warpAffine(
                    rgb_batch[position], d2_affine_matrix(self.epoch, row_index),
                    (d2_size, d2_size), flags=cv2.INTER_LINEAR,
                    borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0)
                )
        return d2_imagenet_input(rgb_batch), self.labels[indices]

    def set_epoch(self, epoch):
        if self.training:
            self.epoch = epoch
            self.order = self.epoch_order()


class D2EpochOrder(keras.callbacks.Callback):
    def __init__(self, train_batches):
        super().__init__()
        self.train_batches = train_batches

    def on_epoch_begin(self, epoch, logs=None):
        self.train_batches.set_epoch(epoch)


def make_d2_batches(mode, split, batch_size=d2_batch_size):
    if mode not in d2_modes or split not in d2_fit_counts or batch_size < 1:
        raise ValueError((mode, split, batch_size))
    images = np.load(d2_cache_dir / f"{mode}.npy", mmap_mode="r", allow_pickle=False)
    assert images.shape == (len(d2_fit_rows), d2_size, d2_size, 3) and images.dtype == np.uint8
    return D2FitBatches(images, d2_fit_labels, d2_fit_indices[split], split == "train", batch_size)


d2_image_root = mounted_dataset("assignment02-d2-rice-leaf-5932") / "Rice Leaf Disease Images"
d2_manifest_path = mounted_dataset("assignment02-verified-train-val-test-splits") / "D2_split_manifest.csv"
d2_fit_rows = [
    row for row in read_d2_manifest(d2_manifest_path) if row["split"] in d2_fit_counts
]
assert Counter(row["split"] for row in d2_fit_rows) == d2_fit_counts
d2_fit_labels = np.asarray([int(row["class_id"]) for row in d2_fit_rows], dtype=np.int32)
d2_fit_indices = {
    split: np.asarray([i for i, row in enumerate(d2_fit_rows) if row["split"] == split], dtype=np.int64)
    for split in d2_fit_counts
}
print("Preparing D2 train and validation images in both modes.")
build_d2_fit_cache(d2_image_root, d2_fit_rows)
print(f"Cached per mode: {len(d2_fit_rows):,} RGB images | {d2_fit_counts}")

pixel = np.array([[[[10, 20, 30]]]], dtype=np.uint8)
np.testing.assert_array_equal(
    d2_imagenet_input(pixel), keras.applications.resnet.preprocess_input(pixel.astype(np.float32))
)

train_pair = [make_d2_batches(mode, "train") for mode in d2_modes]
val_pair = [make_d2_batches(mode, "val") for mode in d2_modes]
assert all(len(batches) == 111 for batches in train_pair)
assert all(len(batches) == 24 for batches in val_pair)
assert all(len(batches.batch_rows(110)) == 27 for batches in train_pair)
assert all(len(batches.batch_rows(23)) == 22 for batches in val_pair)

for split, pair in (("train", train_pair), ("val", val_pair)):
    np.testing.assert_array_equal(pair[0].order, pair[1].order)
    np.testing.assert_array_equal(np.sort(pair[0].order), d2_fit_indices[split])
    x_otsu, y_otsu = pair[0][0]
    x_stretch, y_stretch = pair[1][0]
    np.testing.assert_array_equal(y_otsu, y_stretch)
    assert x_otsu.shape == x_stretch.shape == (30, 224, 224, 3)
    assert x_otsu.dtype == x_stretch.dtype == np.float32
    assert np.isfinite(x_otsu).all() and np.isfinite(x_stretch).all()
    assert np.any(x_otsu != x_stretch), split
    np.testing.assert_array_equal(pair[0][0][0], x_otsu)
    print(f"{split}: {len(pair[0])} batches | paired labels verified | first batch {x_otsu.shape}")

first_order = train_pair[0].order.copy()
first_row = first_order[0]
first_matrix = d2_affine_matrix(0, first_row)
for pair in (train_pair, val_pair):
    for batches in pair:
        batches.set_epoch(1)
np.testing.assert_array_equal(train_pair[0].order, train_pair[1].order)
np.testing.assert_array_equal(np.sort(train_pair[0].order), d2_fit_indices["train"])
assert not np.array_equal(first_order, train_pair[0].order)
assert not np.array_equal(first_matrix, d2_affine_matrix(1, first_row))
np.testing.assert_array_equal(val_pair[0].order, d2_fit_indices["val"])
np.testing.assert_array_equal(val_pair[0][0][0], x_otsu)
print("Epoch 2: paired training order verified, fresh affine draws, validation unchanged.")
print(f"Cache: {d2_cache_dir}")
print("D2 PAIRED INPUT PIPELINE PASSED")


Preparing D2 train and validation images in both modes.
Cached per mode: 4,039 RGB images | {'train': 3327, 'val': 712}
train: 111 batches | paired labels verified | first batch (30, 224, 224, 3)
val: 24 batches | paired labels verified | first batch (30, 224, 224, 3)
Epoch 2: paired training order verified, fresh affine draws, validation unchanged.
Cache: /kaggle/working/paper5_d2_preprocessed
D2 PAIRED INPUT PIPELINE PASSED


In [6]:
import hashlib
import json
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter

import numpy as np
import tensorflow as tf
from tensorflow import keras

d2_train_seed = 2026
d2_train_batch_size = 30
d2_learning_rate = 1e-4
d2_momentum = 0.9
d2_train_device = "/GPU:0"
d2_memory_device = "GPU:0"
d2_init_dir = Path("/kaggle/working/paper5_d2_initialization")
d2_smoke_dir = Path("/kaggle/working/paper5_d2_smoke")


def d2_file_sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def d2_verified_initialization():
    metadata = json.loads((d2_init_dir / "D2_imagenet_transfer.json").read_text(encoding="utf-8"))
    path = d2_init_dir / "D2_amff_shared_init.weights.h5"
    assert metadata["dataset"] == "D2" and metadata["checkpoint"] == path.name, metadata
    actual_hash = d2_file_sha256(path)
    assert actual_hash == metadata["checkpoint_sha256"], "D2 initialization checkpoint hash mismatch"
    return path, actual_hash


def make_d2_optimizer():
    return keras.optimizers.SGD(
        learning_rate=d2_learning_rate, momentum=d2_momentum, nesterov=False
    )


def run_d2_training_smoke(initialization, initialization_hash):
    keras.mixed_precision.set_global_policy("float32")
    keras.utils.set_random_seed(d2_train_seed)
    tf.config.experimental.enable_op_determinism()
    assert tf.config.list_logical_devices("GPU"), "Enable a GPU in the D2 Kaggle session."
    batches = make_d2_batches("otsu", "train", batch_size=d2_train_batch_size)
    d2_smoke_dir.mkdir(parents=True, exist_ok=True)
    tf.config.experimental.reset_memory_stats(d2_memory_device)
    print(f"Training smoke test: {d2_train_device} | float32 | batch {d2_train_batch_size}")
    print(f"Shared initialization SHA-256: {initialization_hash}")

    with tf.device(d2_train_device):
        model = build_amff(4)
        model.load_weights(str(initialization))
        optimizer = make_d2_optimizer()
        loss_fn = keras.losses.SparseCategoricalCrossentropy()
        variables = model.trainable_variables
        assert len(variables) == 94
        head = model.get_layer("class_probabilities")
        initial_kernel = head.kernel.numpy()
        optimizer.build(variables)

        @tf.function(jit_compile=False)
        def train_step(images, labels):
            with tf.GradientTape() as tape:
                probabilities = model(images, training=True)
                loss = loss_fn(labels, probabilities)
            gradients = tape.gradient(loss, variables)
            if any(gradient is None for gradient in gradients):
                raise RuntimeError("A trainable variable has no gradient.")
            loss = tf.debugging.check_numerics(loss, "Nonfinite training loss")
            gradient_norm = tf.debugging.check_numerics(
                tf.linalg.global_norm(gradients), "Nonfinite global gradient norm"
            )
            tf.debugging.assert_positive(gradient_norm)
            optimizer.apply_gradients(zip(gradients, variables))
            return loss, gradient_norm

        measurements = []
        for index in range(3):
            started = perf_counter()
            images, labels = batches[index]
            assert images.shape[0] == d2_train_batch_size
            loss, norm = train_step(tf.convert_to_tensor(images), tf.convert_to_tensor(labels))
            loss_value = float(loss.numpy())
            norm_value = float(norm.numpy())
            assert int(optimizer.iterations.numpy()) == index + 1
            elapsed = perf_counter() - started
            measurements.append({
                "step": index + 1, "seconds": elapsed, "loss": loss_value,
                "global_gradient_norm": norm_value
            })
            print(f"Step {index + 1}/3: loss {loss_value:.5f} | gradient norm {norm_value:.5f} | {elapsed:.2f} s")

        assert not np.array_equal(initial_kernel, head.kernel.numpy()), "Classifier weights did not update"
        peak = tf.config.experimental.get_memory_info(d2_memory_device)["peak"]
        rate = 2 * d2_train_batch_size / sum(item["seconds"] for item in measurements[1:])
        print(f"TensorFlow peak device memory: {peak / 2**30:.2f} GiB")
        print(f"Throughput after warm-up: {rate:.1f} images/s")

        probe = tf.convert_to_tensor(images[:1])
        expected = model(probe, training=False).numpy()
        assert expected.shape == (1, 4) and np.isfinite(expected).all()
        expected_bias = head.bias.numpy()
        with TemporaryDirectory(dir=d2_smoke_dir) as scratch:
            scratch_path = Path(scratch) / "disposable.weights.h5"
            model.save_weights(str(scratch_path))
            head.bias.assign_add(tf.one_hot(0, 4, dtype=tf.float32))
            assert not np.array_equal(head.bias.numpy(), expected_bias)
            model.load_weights(str(scratch_path))
            np.testing.assert_array_equal(head.bias.numpy(), expected_bias)
            actual = model(probe, training=False).numpy()
            np.testing.assert_allclose(actual, expected, rtol=1e-6, atol=1e-7)
        print("Checkpoint round trip passed. Disposable weights cleared.")

    assert d2_file_sha256(initialization) == initialization_hash, "Shared initialization changed"
    report = {
        "status": "PASS", "dataset": "D2", "mode": "otsu", "seed": d2_train_seed,
        "tensorflow": tf.__version__, "keras": keras.__version__, "device": d2_train_device,
        "dtype": "float32", "op_determinism": True, "jit_compile": False,
        "batch_size": d2_train_batch_size, "learning_rate": d2_learning_rate,
        "optimizer": "SGD", "momentum": d2_momentum, "nesterov": False,
        "initialization_sha256": initialization_hash, "trainable_variables_checked": len(variables),
        "steps": measurements, "peak_device_bytes": peak, "images_per_second_after_warmup": rate,
        "checkpoint_round_trip": True
    }
    path = d2_smoke_dir / "D2_training_smoke.json"
    path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    print(f"Smoke report: {path}")


globals().pop("d2_amff", None)
keras.backend.clear_session()
initialization_path, initialization_sha256 = d2_verified_initialization()
run_d2_training_smoke(initialization_path, initialization_sha256)
keras.backend.clear_session()
print("D2 TRAINING SMOKE TEST PASSED")


Training smoke test: /GPU:0 | float32 | batch 30
Shared initialization SHA-256: 545cade231e29e4c552ef469e0d0205d567fb2efa4b82e5fe720715f9a9e0506
Step 1/3: loss 16.81354 | gradient norm 1047.91772 | 5.72 s
Step 2/3: loss 17.12397 | gradient norm 874.93781 | 0.41 s
Step 3/3: loss 10.71786 | gradient norm 627.03528 | 0.40 s
TensorFlow peak device memory: 3.47 GiB
Throughput after warm-up: 73.7 images/s
Checkpoint round trip passed. Disposable weights cleared.
Smoke report: /kaggle/working/paper5_d2_smoke/D2_training_smoke.json
D2 TRAINING SMOKE TEST PASSED


In [7]:
import json
import platform
from pathlib import Path
from tempfile import mkdtemp
from time import perf_counter

import cv2
import numpy as np
import tensorflow as tf
from PIL import __version__ as pillow_version
from tensorflow import keras

d2_epochs = 6
d2_run_root = Path("/kaggle/working/paper5_d2_runs")


def write_d2_json(path, value):
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)


def train_d2_variant(mode):
    initialization, init_hash = d2_verified_initialization()
    smoke_path = d2_smoke_dir / "D2_training_smoke.json"
    smoke = json.loads(smoke_path.read_text(encoding="utf-8"))
    assert smoke["status"] == "PASS" and smoke["checkpoint_round_trip"]
    assert smoke["initialization_sha256"] == init_hash
    assert smoke["batch_size"] == d2_train_batch_size == 30
    assert smoke["learning_rate"] == d2_learning_rate == 1e-4
    assert smoke["momentum"] == d2_momentum == 0.9
    assert smoke["device"] == d2_train_device and smoke["dtype"] == "float32"
    assert smoke["tensorflow"] == tf.__version__ and smoke["keras"] == keras.__version__

    training = make_d2_batches(mode, "train", batch_size=d2_train_batch_size)
    validation = make_d2_batches(mode, "val", batch_size=d2_train_batch_size)
    assert (len(training), len(validation)) == (111, 24)
    keras.backend.clear_session()
    keras.mixed_precision.set_global_policy("float32")
    keras.utils.set_random_seed(d2_train_seed)
    tf.config.experimental.enable_op_determinism()
    gpu = tf.config.list_physical_devices("GPU")[0]
    hardware = tf.config.experimental.get_device_details(gpu)

    d2_run_root.mkdir(parents=True, exist_ok=True)
    run_dir = Path(mkdtemp(prefix=f"{mode}_", dir=d2_run_root))
    checkpoint = run_dir / "best_validation.weights.h5"
    configuration = {
        "dataset": "D2",
        "method": "Paper 5 AMFF-Hybrid Net paper-guided reconstruction",
        "mode": mode,
        "manifest_sha256": d2_file_sha256(d2_manifest_path),
        "initialization_sha256": init_hash,
        "input_shape": [224, 224, 3],
        "class_map": {int(row["class_id"]): row["class_name"] for row in d2_fit_rows},
        "fit_split_counts": d2_fit_counts,
        "epoch_count": d2_epochs,
        "batch_size": d2_train_batch_size,
        "training_batches_per_epoch": len(training),
        "validation_batches": len(validation),
        "include_remainder_batch": True,
        "optimizer": "SGD",
        "learning_rate": d2_learning_rate,
        "momentum": d2_momentum,
        "nesterov": False,
        "weight_decay": None,
        "loss": "sparse categorical cross-entropy",
        "seed": d2_train_seed,
        "data_seed": d2_data_seed,
        "precision": "float32",
        "jit_compile": False,
        "op_determinism": True,
        "checkpoint_selection": "Minimum validation loss with earlier epoch winning ties",
        "validation_frequency_epochs": 1,
        "preprocessing": {
            "resize": "Fit longest side to 224, preserve aspect ratio",
            "contrast": "Per-image CIELAB L* min-max stretch",
            "otsu_bright_mask": mode == "otsu",
            "padding": "Center-pad to 224 with RGB zeros after preprocessing",
            "normalization": "Keras ImageNet RGB-to-BGR and channel mean subtraction"
        },
        "training_augmentation": {
            "translation_pixels": [-25, 25],
            "independent_scale": [0.9, 1.1],
            "rotation_degrees": [-20, 20],
            "interpolation": "bilinear",
            "fill_rgb": [0, 0, 0],
            "seed_inputs": ["data seed", "zero-based epoch", "cache row index"]
        },
        "runtime": {
            "python": platform.python_version(), "tensorflow": tf.__version__,
            "keras": keras.__version__, "numpy": np.__version__,
            "opencv": cv2.__version__, "pillow": pillow_version,
            "device": d2_train_device, "gpu": hardware
        }
    }
    write_d2_json(run_dir / "training_config.json", configuration)
    print(f"D2 {mode} | {d2_epochs} epochs | batch {d2_train_batch_size} | {d2_train_device}")
    print(f"Shared initialization SHA-256: {init_hash}")
    print(f"Run directory: {run_dir}")

    with tf.device(d2_train_device):
        model = build_amff(4)
        model.load_weights(str(initialization))
        model.compile(
            optimizer=make_d2_optimizer(),
            loss=keras.losses.SparseCategoricalCrossentropy(),
            metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
            jit_compile=False
        )
        checkpoint_callback = keras.callbacks.ModelCheckpoint(
            str(checkpoint), monitor="val_loss", mode="min",
            save_best_only=True, save_weights_only=True, verbose=1
        )
        callbacks = [
            D2EpochOrder(training),
            keras.callbacks.TerminateOnNaN(),
            checkpoint_callback,
            keras.callbacks.CSVLogger(str(run_dir / "epoch_log.csv"))
        ]
        started = perf_counter()
        history = model.fit(
            training, validation_data=validation, epochs=d2_epochs,
            validation_freq=1, shuffle=False, verbose=2, callbacks=callbacks
        ).history
        fit_seconds = perf_counter() - started
        metrics = ("loss", "accuracy", "val_loss", "val_accuracy")
        assert all(len(history[key]) == d2_epochs for key in metrics), history
        assert all(np.isfinite(history[key]).all() for key in metrics), history
        updates = int(model.optimizer.iterations.numpy())
        assert updates == len(training) * d2_epochs, updates
        assert checkpoint.is_file(), checkpoint

        best_index = int(np.argmin(history["val_loss"]))
        np.testing.assert_allclose(checkpoint_callback.best, history["val_loss"][best_index])
        model.load_weights(str(checkpoint))
        selected_val = model.evaluate(validation, verbose=0, return_dict=True)
        np.testing.assert_allclose(
            [selected_val["loss"], selected_val["accuracy"]],
            [history["val_loss"][best_index], history["val_accuracy"][best_index]],
            rtol=1e-5, atol=1e-6
        )

    summary = {
        "status": "PASS", "dataset": "D2", "mode": mode,
        "epochs_completed": d2_epochs, "optimizer_updates": updates,
        "fit_seconds": fit_seconds, "initialization_sha256": init_hash,
        "best_epoch": best_index + 1, "best_val_loss": float(selected_val["loss"]),
        "best_val_accuracy": float(selected_val["accuracy"]),
        "best_checkpoint": checkpoint.name, "best_checkpoint_sha256": d2_file_sha256(checkpoint),
        "history": {key: [float(value) for value in history[key]] for key in metrics}
    }
    write_d2_json(run_dir / "training_summary.json", summary)
    print(f"Completed: {d2_epochs} epochs | {updates} optimizer updates | {fit_seconds:.1f} s")
    print(
        f"Best validation checkpoint: epoch {summary['best_epoch']} | "
        f"loss {summary['best_val_loss']:.6f} | accuracy {summary['best_val_accuracy']:.4%}"
    )
    print(f"Checkpoint SHA-256: {summary['best_checkpoint_sha256']}")
    print(f"Training summary: {run_dir / 'training_summary.json'}")
    keras.backend.clear_session()
    return run_dir


d2_otsu_run_dir = train_d2_variant("otsu")
print("D2 PRIMARY OTSU TRAINING PASSED")


D2 otsu | 6 epochs | batch 30 | /GPU:0
Shared initialization SHA-256: 545cade231e29e4c552ef469e0d0205d567fb2efa4b82e5fe720715f9a9e0506
Run directory: /kaggle/working/paper5_d2_runs/otsu_hd7zh1yp
Epoch 1/6

Epoch 1: val_loss improved from None to 0.66813, saving model to /kaggle/working/paper5_d2_runs/otsu_hd7zh1yp/best_validation.weights.h5

Epoch 1: finished saving model to /kaggle/working/paper5_d2_runs/otsu_hd7zh1yp/best_validation.weights.h5
111/111 - 61s - 550ms/step - accuracy: 0.5717 - loss: 1.9948 - val_accuracy: 0.7303 - val_loss: 0.6681
Epoch 2/6

Epoch 2: val_loss improved from 0.66813 to 0.51524, saving model to /kaggle/working/paper5_d2_runs/otsu_hd7zh1yp/best_validation.weights.h5

Epoch 2: finished saving model to /kaggle/working/paper5_d2_runs/otsu_hd7zh1yp/best_validation.weights.h5
111/111 - 54s - 484ms/step - accuracy: 0.7298 - loss: 0.6691 - val_accuracy: 0.7865 - val_loss: 0.5152
Epoch 3/6

Epoch 3: val_loss did not improve from 0.51524
111/111 - 52s - 471ms/step -

In [8]:
d2_primary_summary = json.loads(
    (d2_otsu_run_dir / "training_summary.json").read_text(encoding="utf-8")
)
assert d2_primary_summary["status"] == "PASS" and d2_primary_summary["mode"] == "otsu"
d2_primary_checkpoint = d2_otsu_run_dir / d2_primary_summary["best_checkpoint"]
d2_primary_hash = d2_file_sha256(d2_primary_checkpoint)
assert d2_primary_hash == d2_primary_summary["best_checkpoint_sha256"]

d2_stretch_run_dir = train_d2_variant("stretch_only")

d2_primary_config = json.loads(
    (d2_otsu_run_dir / "training_config.json").read_text(encoding="utf-8")
)
d2_comparison_config = json.loads(
    (d2_stretch_run_dir / "training_config.json").read_text(encoding="utf-8")
)
assert d2_primary_config.pop("mode") == "otsu"
assert d2_comparison_config.pop("mode") == "stretch_only"
assert d2_primary_config["preprocessing"].pop("otsu_bright_mask") is True
assert d2_comparison_config["preprocessing"].pop("otsu_bright_mask") is False
assert d2_primary_config == d2_comparison_config, "The paired training settings differ"
assert d2_file_sha256(d2_primary_checkpoint) == d2_primary_hash

print("Matched training settings verified.")
print("Primary Otsu checkpoint hash verified.")
print("D2 STRETCH-ONLY COMPARISON TRAINING PASSED")

D2 stretch_only | 6 epochs | batch 30 | /GPU:0
Shared initialization SHA-256: 545cade231e29e4c552ef469e0d0205d567fb2efa4b82e5fe720715f9a9e0506
Run directory: /kaggle/working/paper5_d2_runs/stretch_only_v2odbrhu
Epoch 1/6

Epoch 1: val_loss improved from None to 0.69979, saving model to /kaggle/working/paper5_d2_runs/stretch_only_v2odbrhu/best_validation.weights.h5

Epoch 1: finished saving model to /kaggle/working/paper5_d2_runs/stretch_only_v2odbrhu/best_validation.weights.h5
111/111 - 61s - 552ms/step - accuracy: 0.6561 - loss: 1.6358 - val_accuracy: 0.7598 - val_loss: 0.6998
Epoch 2/6

Epoch 2: val_loss improved from 0.69979 to 0.49509, saving model to /kaggle/working/paper5_d2_runs/stretch_only_v2odbrhu/best_validation.weights.h5

Epoch 2: finished saving model to /kaggle/working/paper5_d2_runs/stretch_only_v2odbrhu/best_validation.weights.h5
111/111 - 53s - 477ms/step - accuracy: 0.8236 - loss: 0.4656 - val_accuracy: 0.8315 - val_loss: 0.4951
Epoch 3/6

Epoch 3: val_loss improved 

In [9]:
import csv
import json
import os
from zipfile import ZIP_DEFLATED, ZipFile

import numpy as np
import tensorflow as tf
from IPython.display import FileLink, display
from tensorflow import keras


def load_d2_selected_model(run_dir, mode):
    summary = json.loads((run_dir / "training_summary.json").read_text(encoding="utf-8"))
    assert summary["status"] == "PASS" and summary["dataset"] == "D2" and summary["mode"] == mode
    assert summary["epochs_completed"] == 6 and summary["optimizer_updates"] == 666
    checkpoint = run_dir / summary["best_checkpoint"]
    actual_hash = d2_file_sha256(checkpoint)
    assert actual_hash == summary["best_checkpoint_sha256"], (mode, "Checkpoint hash mismatch")

    keras.backend.clear_session()
    keras.mixed_precision.set_global_policy("float32")
    keras.utils.set_random_seed(d2_train_seed)
    with tf.device(d2_train_device):
        model = build_amff(4)
        model.load_weights(str(checkpoint))
    return model, summary


def d2_classification_metrics(truth, probabilities):
    predictions = probabilities.argmax(axis=1)
    confusion = np.zeros((4, 4), dtype=np.int64)
    np.add.at(confusion, (truth, predictions), 1)
    support = confusion.sum(axis=1)
    predicted_count = confusion.sum(axis=0)
    correct = np.diag(confusion)
    assert support.min() > 0
    precision = np.divide(correct, predicted_count, out=np.zeros(4), where=predicted_count > 0)
    recall = correct / support
    f1 = 2 * correct / (support + predicted_count)
    return {
        "correct": int(correct.sum()),
        "accuracy": float(correct.sum() / len(truth)),
        "macro_precision": float(precision.mean()),
        "macro_recall": float(recall.mean()),
        "macro_f1": float(f1.mean()),
        "confusion_matrix": confusion.tolist(),
        "per_class": [
            {
                "class_id": i, "support": int(support[i]), "correct": int(correct[i]),
                "precision": float(precision[i]), "recall": float(recall[i]), "f1": float(f1[i])
            }
            for i in range(4)
        ]
    }


def predict_d2_validation(run_dir, mode, validation_indices):
    model, summary = load_d2_selected_model(run_dir, mode)
    batches = make_d2_batches(mode, "val", batch_size=d2_train_batch_size)
    np.testing.assert_array_equal(batches.order, validation_indices)
    observed_labels = []
    observed_probabilities = []
    with tf.device(d2_train_device):
        for index in range(len(batches)):
            images, labels = batches[index]
            observed_labels.append(labels)
            observed_probabilities.append(model(images, training=False).numpy())

    truth = np.concatenate(observed_labels)
    probabilities = np.concatenate(observed_probabilities)
    assert truth.shape == (712,) and probabilities.shape == (712, 4)
    assert np.isfinite(probabilities).all() and np.all((probabilities >= 0) & (probabilities <= 1))
    np.testing.assert_allclose(probabilities.sum(axis=1), 1.0, rtol=0, atol=1e-5)
    np.testing.assert_array_equal(truth, d2_fit_labels[validation_indices])
    metrics = d2_classification_metrics(truth, probabilities)
    np.testing.assert_allclose(metrics["accuracy"], summary["best_val_accuracy"], rtol=0, atol=1e-7)
    metrics["checkpoint_sha256"] = summary["best_checkpoint_sha256"]
    metrics["initialization_sha256"] = summary["initialization_sha256"]
    metrics["selected_epoch"] = summary["best_epoch"]
    metrics["validation_loss"] = summary["best_val_loss"]
    return probabilities, metrics


def save_d2_validation_predictions(path, rows, probabilities):
    fields = ["rel_path", "true_class_id", "predicted_class_id"] + [f"p_{i}" for i in range(4)]
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(fields)
        for row, probability in zip(rows, probabilities, strict=True):
            writer.writerow([
                row["rel_path"], row["class_id"], int(probability.argmax()),
                *[f"{value:.9g}" for value in probability]
            ])


def run_d2_validation_comparison():
    validation_indices = d2_fit_indices["val"]
    rows = [d2_fit_rows[i] for i in validation_indices]
    assert len({row["rel_path"] for row in rows}) == 712
    assert {row["split"] for row in rows} == {"val"}
    class_names = {int(row["class_id"]): row["class_name"] for row in rows}
    assert set(class_names) == set(range(4))

    probabilities = {}
    results = {}
    for mode, run_dir in (("otsu", d2_otsu_run_dir), ("stretch_only", d2_stretch_run_dir)):
        probabilities[mode], results[mode] = predict_d2_validation(run_dir, mode, validation_indices)
        scores = results[mode]
        print(
            f"{mode}: epoch {scores['selected_epoch']} | "
            f"{scores['correct']}/712 correct | accuracy {scores['accuracy']:.4%} | "
            f"macro F1 {scores['macro_f1']:.6f} | val loss {scores['validation_loss']:.6f}"
        )
    primary, comparison = (results[mode] for mode in ("otsu", "stretch_only"))
    assert primary["initialization_sha256"] == comparison["initialization_sha256"]

    comparison_rows = []
    print("Class | support | Otsu recall | Stretch-only recall | change (pp)")
    for class_id, (baseline, alternative) in enumerate(zip(primary["per_class"], comparison["per_class"], strict=True)):
        assert baseline["class_id"] == alternative["class_id"] == class_id
        assert baseline["support"] == alternative["support"]
        label = class_names[class_id]
        delta = 100 * (alternative["recall"] - baseline["recall"])
        comparison_rows.append({
            "class_id": class_id, "class_name": label, "support": baseline["support"],
            "otsu_correct": baseline["correct"], "stretch_only_correct": alternative["correct"],
            "otsu_recall": baseline["recall"], "stretch_only_recall": alternative["recall"],
            "recall_change_percentage_points": delta
        })
        print(
            f"{label} | {baseline['support']} | {baseline['recall']:.2%} | "
            f"{alternative['recall']:.2%} | {delta:+.2f}"
        )

    delta_f1 = 100 * (comparison["macro_f1"] - primary["macro_f1"])
    print(f"Stretch-only minus Otsu validation macro F1: {delta_f1:+.2f} percentage points")
    export_dir = d2_run_root / "validation_comparison"
    export_dir.mkdir(parents=True, exist_ok=True)
    files = []
    for mode in ("otsu", "stretch_only"):
        path = export_dir / f"D2_{mode}_validation_predictions.csv"
        save_d2_validation_predictions(path, rows, probabilities[mode])
        files.append(path)

    recalls = export_dir / "D2_validation_recall_comparison.csv"
    with recalls.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=comparison_rows[0].keys())
        writer.writeheader()
        writer.writerows(comparison_rows)
    files.append(recalls)

    summary_path = export_dir / "D2_validation_comparison.json"
    write_d2_json(summary_path, {
        "dataset": "D2", "split": "val", "image_count": len(rows), "primary_mode": "otsu",
        "manifest_sha256": d2_file_sha256(d2_manifest_path), "class_map": class_names,
        "runs": results, "stretch_only_minus_otsu_macro_f1_percentage_points": delta_f1
    })
    files.append(summary_path)

    archive_path = export_dir / "D2_Step_9_Validation_Comparison.zip"
    with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
        for path in files:
            archive.write(path, arcname=path.name)
    with ZipFile(archive_path) as archive:
        assert len(archive.namelist()) == 4 and archive.testzip() is None
    print("D2 PAIRED VALIDATION COMPARISON PASSED")
    display(FileLink(os.path.relpath(archive_path)))
    return export_dir


d2_validation_comparison_dir = run_d2_validation_comparison()


otsu: epoch 6 | 589/712 correct | accuracy 82.7247% | macro F1 0.807922 | val loss 0.397404
stretch_only: epoch 6 | 636/712 correct | accuracy 89.3258% | macro F1 0.886794 | val loss 0.323818
Class | support | Otsu recall | Stretch-only recall | change (pp)
Bacterial Blight | 192 | 77.60% | 83.85% | +6.25
Blast | 144 | 54.86% | 80.56% | +25.69
Brown Spot | 180 | 92.22% | 92.78% | +0.56
Tungro | 196 | 99.49% | 97.96% | -1.53
Stretch-only minus Otsu validation macro F1: +7.89 percentage points
D2 PAIRED VALIDATION COMPARISON PASSED


/kaggle/working/paper5_d2_runs/validation_comparison/D2_Step_9_Validation_Comparison.zip

In [10]:
import csv
import json
import os
from textwrap import fill
from zipfile import ZIP_DEFLATED, ZipFile

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import sklearn
import tensorflow as tf
from IPython.display import FileLink, display
from sklearn.metrics import auc, roc_auc_score, roc_curve

d2_locked_otsu_hash = "f8be8fc49c505731b3a95e91ebbdaed299625b6fa8a5bfb2d024b8b9282cb3d3"


def load_or_predict_d2_test(rows, prediction_path):
    truth = np.asarray([int(row["class_id"]) for row in rows], dtype=np.int32)
    if prediction_path.is_file():
        with prediction_path.open(newline="", encoding="utf-8") as stream:
            saved = list(csv.DictReader(stream))
        assert [row["rel_path"] for row in saved] == [row["rel_path"] for row in rows]
        np.testing.assert_array_equal([int(row["true_class_id"]) for row in saved], truth)
        probabilities = np.asarray([[row[f"p_{i}"] for i in range(4)] for row in saved], dtype=np.float32)
        np.testing.assert_array_equal([int(row["predicted_class_id"]) for row in saved], probabilities.argmax(axis=1))
        print("Reusing the existing test predictions for this locked checkpoint.")
    else:
        model, summary = load_d2_selected_model(d2_otsu_run_dir, "otsu")
        assert summary["best_checkpoint_sha256"] == d2_locked_otsu_hash
        output = []
        with tf.device(d2_train_device):
            for start in range(0, len(rows), d2_train_batch_size):
                batch = rows[start:start + d2_train_batch_size]
                rgb = np.stack([prepare_d2_image(d2_image_root / row["rel_path"])[0] for row in batch])
                output.append(model(d2_imagenet_input(rgb), training=False).numpy())
        probabilities = np.concatenate(output)

    assert probabilities.shape == (713, 4)
    assert np.isfinite(probabilities).all() and np.all((probabilities >= 0) & (probabilities <= 1))
    np.testing.assert_allclose(probabilities.sum(axis=1), 1.0, rtol=0, atol=1e-5)
    if not prediction_path.exists():
        temporary = prediction_path.with_suffix(".tmp")
        save_d2_validation_predictions(temporary, rows, probabilities)
        temporary.replace(prediction_path)
    return truth, probabilities


def plot_d2_test_confusion(path, confusion, names):
    figure, axis = plt.subplots(figsize=(9, 8), layout="constrained")
    matrix = axis.imshow(confusion, cmap="Blues")
    axis.set_xticks(range(4), names, rotation=38, ha="right")
    axis.set_yticks(range(4), names)
    axis.set(xlabel="Predicted class", ylabel="True class", title="D2 held-out test confusion matrix · AMFF-Hybrid Net + Otsu")
    for (i, j), count in np.ndenumerate(confusion):
        color = "white" if count > confusion.max() / 2 else "black"
        axis.text(j, i, str(count), ha="center", va="center", color=color)
    figure.colorbar(matrix, ax=axis, shrink=0.75, label="Image count")
    figure.savefig(path, dpi=180)
    plt.close(figure)


def plot_d2_test_roc(path, truth, probabilities, names, macro_auc):
    figure, axis = plt.subplots(figsize=(9, 5), layout="constrained")
    for i, name in enumerate(names):
        fpr, tpr, _ = roc_curve(truth == i, probabilities[:, i])
        axis.plot(fpr, tpr, linewidth=1.8, label=f"{name} ({auc(fpr, tpr):.3f})")
    axis.plot([0, 1], [0, 1], color="0.45", linestyle="--", linewidth=1)
    axis.set(
        xlabel="False positive rate", ylabel="True positive rate", xlim=(0, 1), ylim=(0, 1.02),
        title=f"D2 held-out one-vs-rest ROC · Macro AUC {macro_auc:.3f}"
    )
    axis.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=9, title="Class (AUC)")
    figure.savefig(path, dpi=180)
    plt.close(figure)


def plot_d2_training_history(path, summary):
    history = summary["history"]
    epochs = np.arange(1, len(history["loss"]) + 1)
    figure, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
    for axis, key, label in zip(axes, ("loss", "accuracy"), ("Cross-entropy loss", "Accuracy"), strict=True):
        axis.plot(epochs, history[key], marker="o", label="Training")
        axis.plot(epochs, history[f"val_{key}"], marker="o", label="Validation")
        axis.axvline(summary["best_epoch"], color="0.45", linestyle="--", label="Selected epoch")
        axis.set(xlabel="Epoch", ylabel=label, xticks=epochs)
        axis.legend(fontsize=8)
        axis.grid(alpha=0.2)
    figure.suptitle("D2 primary Otsu training history")
    figure.savefig(path, dpi=180)
    plt.close(figure)


def plot_d2_test_examples(path, rows, truth, probabilities, names):
    predicted = probabilities.argmax(axis=1)
    correct = np.flatnonzero(predicted == truth)[:2]
    incorrect = np.flatnonzero(predicted != truth)[:2]
    indices = np.concatenate([correct, incorrect])
    figure, axes = plt.subplots(len(indices), 2, figsize=(7, 3 * len(indices)), layout="constrained", squeeze=False)
    for row_axes, index in zip(axes, indices, strict=True):
        masked, stretched = prepare_d2_image(d2_image_root / rows[index]["rel_path"])
        actual_name = fill(names[truth[index]], 23)
        predicted_name = fill(names[predicted[index]], 23)
        row_axes[0].imshow(stretched)
        row_axes[0].set_title(f"Before Otsu\nTrue: {actual_name}")
        row_axes[1].imshow(masked)
        row_axes[1].set_title(f"Model input\nPredicted: {predicted_name}")
        for axis in row_axes:
            axis.axis("off")
    figure.suptitle("D2 test examples · First two correct and first two errors in manifest order")
    figure.savefig(path, dpi=160)
    plt.close(figure)
    return [rows[i]["rel_path"] for i in indices]


def run_d2_locked_otsu_test():
    summary = json.loads((d2_otsu_run_dir / "training_summary.json").read_text(encoding="utf-8"))
    assert summary["status"] == "PASS" and summary["mode"] == "otsu" and summary["best_epoch"] == 6
    checkpoint = d2_otsu_run_dir / summary["best_checkpoint"]
    assert d2_file_sha256(checkpoint) == summary["best_checkpoint_sha256"] == d2_locked_otsu_hash
    rows = [row for row in read_d2_manifest(d2_manifest_path) if row["split"] == "test"]
    assert len(rows) == len({row["rel_path"] for row in rows}) == 713
    class_map = {int(row["class_id"]): row["class_name"] for row in rows}
    names = [class_map[i] for i in range(4)]
    print(f"Locked primary Otsu checkpoint: epoch {summary['best_epoch']} | SHA-256 {d2_locked_otsu_hash}")

    export_dir = d2_otsu_run_dir / "held_out_test"
    export_dir.mkdir(parents=True, exist_ok=True)
    prediction_path = export_dir / "D2_otsu_test_predictions.csv"
    truth, probabilities = load_or_predict_d2_test(rows, prediction_path)
    metrics = d2_classification_metrics(truth, probabilities)
    metrics["macro_ovr_auc"] = float(roc_auc_score(truth, probabilities, multi_class="ovr", average="macro"))
    metrics["weighted_ovr_auc"] = float(roc_auc_score(truth, probabilities, multi_class="ovr", average="weighted"))
    for entry in metrics["per_class"]:
        i = entry["class_id"]
        entry["class_name"] = names[i]
        entry["ovr_auc"] = float(roc_auc_score(truth == i, probabilities[:, i]))
    np.testing.assert_allclose(metrics["macro_ovr_auc"], np.mean([row["ovr_auc"] for row in metrics["per_class"]]))

    per_class_path = export_dir / "D2_test_per_class.csv"
    with per_class_path.open("w", newline="", encoding="utf-8") as stream:
        fields = ("class_id", "class_name", "support", "correct", "precision", "recall", "f1", "ovr_auc")
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(metrics["per_class"])

    confusion = export_dir / "D2_test_confusion.png"
    roc = export_dir / "D2_test_roc.png"
    curves = export_dir / "D2_training_curves.png"
    examples = export_dir / "D2_test_examples.png"
    plot_d2_test_confusion(confusion, np.asarray(metrics["confusion_matrix"]), names)
    plot_d2_test_roc(roc, truth, probabilities, names, metrics["macro_ovr_auc"])
    plot_d2_training_history(curves, summary)
    sample_paths = plot_d2_test_examples(examples, rows, truth, probabilities, names)
    report_path = export_dir / "D2_test_summary.json"
    write_d2_json(report_path, {
        "dataset": "D2", "split": "test", "mode": "otsu", "image_count": len(rows),
        "checkpoint_epoch": summary["best_epoch"], "checkpoint_sha256": d2_locked_otsu_hash,
        "initialization_sha256": summary["initialization_sha256"],
        "manifest_sha256": d2_file_sha256(d2_manifest_path),
        "predictions_sha256": d2_file_sha256(prediction_path),
        "probability_class_order": names, "examples": sample_paths,
        "sklearn": sklearn.__version__, "matplotlib": matplotlib.__version__, "metrics": metrics
    })

    print(f"Accuracy: {metrics['accuracy']:.4%} ({metrics['correct']}/713)")
    for key in ("macro_precision", "macro_recall", "macro_f1", "macro_ovr_auc", "weighted_ovr_auc"):
        print(f"{key}: {metrics[key]:.6f}")

    archive_path = export_dir / "D2_Step_10_Test_Evidence.zip"
    with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
        for path in (prediction_path, per_class_path, report_path, confusion, roc, curves, examples):
            archive.write(path, arcname=path.name)
        for name in ("training_config.json", "training_summary.json", "epoch_log.csv"):
            archive.write(d2_otsu_run_dir / name, arcname=name)
    with ZipFile(archive_path) as archive:
        assert len(archive.namelist()) == 10 and archive.testzip() is None
    print("D2 PRIMARY OTSU HELD-OUT EVALUATION PASSED")
    display(FileLink(os.path.relpath(archive_path)))
    return export_dir


d2_test_evidence_dir = run_d2_locked_otsu_test()


Locked primary Otsu checkpoint: epoch 6 | SHA-256 f8be8fc49c505731b3a95e91ebbdaed299625b6fa8a5bfb2d024b8b9282cb3d3
Accuracy: 83.8710% (598/713)
macro_precision: 0.830469
macro_recall: 0.830993
macro_f1: 0.828378
macro_ovr_auc: 0.963753
weighted_ovr_auc: 0.966710
D2 PRIMARY OTSU HELD-OUT EVALUATION PASSED


/kaggle/working/paper5_d2_runs/otsu_hd7zh1yp/held_out_test/D2_Step_10_Test_Evidence.zip

In [11]:
import hashlib
import json
import os
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

from IPython.display import FileLink, display

working = Path("/kaggle/working")
runs = working / "paper5_d2_runs"
otsu = runs / "otsu_hd7zh1yp"
stretch = runs / "stretch_only_v2odbrhu"
initialization = working / "paper5_d2_initialization"
manifest_slug = "assignment02-verified-train-val-test-splits"
manifest_locations = [
    Path("/kaggle/input/datasets/nayebul0") / manifest_slug / "D2_split_manifest.csv",
    Path("/kaggle/input") / manifest_slug / "D2_split_manifest.csv"
]
manifest = next((path for path in manifest_locations if path.is_file()), manifest_locations[0])

checkpoints = (
    (
        "D2_Otsu_Epoch6.weights.h5",
        otsu / "best_validation.weights.h5",
        "f8be8fc49c505731b3a95e91ebbdaed299625b6fa8a5bfb2d024b8b9282cb3d3"
    ),
    (
        "D2_Stretch_Only_Epoch6.weights.h5",
        stretch / "best_validation.weights.h5",
        "870dada3526eaa8738db6f07349f1486cec0afa6e99a259d83704048e5bbfd71"
    ),
    (
        "D2_amff_shared_init.weights.h5",
        initialization / "D2_amff_shared_init.weights.h5",
        "545cade231e29e4c552ef469e0d0205d567fb2efa4b82e5fe720715f9a9e0506"
    )
)

evidence = {
    "D2_split_manifest.csv": manifest,
    "D2_Step_2_Otsu_Audit.zip": working / "D2_Step_2_Otsu_Audit.zip",
    "D2_imagenet_transfer.json": initialization / "D2_imagenet_transfer.json",
    "D2_training_smoke.json": working / "paper5_d2_smoke/D2_training_smoke.json",
    "D2_Step_9_Validation_Comparison.zip": runs / "validation_comparison/D2_Step_9_Validation_Comparison.zip",
    "D2_Step_10_Test_Evidence.zip": otsu / "held_out_test/D2_Step_10_Test_Evidence.zip"
}
for mode, run_dir in (("otsu", otsu), ("stretch_only", stretch)):
    for name in ("training_config.json", "training_summary.json", "epoch_log.csv"):
        evidence[f"{mode}/{name}"] = run_dir / name


def file_sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


checkpoint_inventory = []
for download_name, path, expected in checkpoints:
    if not path.is_file():
        print(f"MISSING checkpoint: {path}")
        checkpoint_inventory.append({"download_name": download_name, "status": "MISSING", "expected_sha256": expected})
        continue
    actual = file_sha256(path)
    status = "PASS" if actual == expected else "HASH_MISMATCH"
    print(f"{status} {download_name} | {path.stat().st_size / 2**20:.1f} MiB | {actual}")
    display(FileLink(os.path.relpath(path), result_html_prefix=f"Save as {download_name} — "))
    checkpoint_inventory.append({
        "download_name": download_name, "status": status, "bytes": path.stat().st_size,
        "sha256": actual, "expected_sha256": expected
    })

output = working / "D2_Paper5_Evidence_Backup.zip"
temporary = output.with_suffix(".tmp")
included = []
missing = []
with ZipFile(temporary, "w", compression=ZIP_DEFLATED, compresslevel=1) as archive:
    for name, path in evidence.items():
        if not path.is_file():
            missing.append(name)
            continue
        digest = file_sha256(path)
        if name == "D2_split_manifest.csv":
            assert digest == "261b97b6fa700197f2dde6bc0f833bf802bd1301c0611606400e990cfad2a185"
        archive.write(path, arcname=name)
        included.append({"name": name, "bytes": path.stat().st_size, "sha256": digest})

    inventory = {
        "experiment": "Paper 5 AMFF-Hybrid Net reconstruction on D2",
        "dataset_source": "https://data.mendeley.com/datasets/fwcj7stb8r/2",
        "included_evidence": included, "missing_evidence": missing,
        "checkpoints_to_download_separately": checkpoint_inventory
    }
    archive.writestr("D2_backup_inventory.json", json.dumps(inventory, indent=2) + "\n")

with ZipFile(temporary) as archive:
    assert archive.testzip() is None and len(archive.namelist()) == len(included) + 1
temporary.replace(output)
print(f"Evidence archive: {len(included)} files plus inventory | {output.stat().st_size / 2**20:.1f} MiB")
if missing:
    print("Missing evidence:", ", ".join(missing))
if not missing and all(item["status"] == "PASS" for item in checkpoint_inventory):
    print("D2 BACKUP READY: download the evidence ZIP and three checkpoints.")
display(FileLink(os.path.relpath(output)))


PASS D2_Otsu_Epoch6.weights.h5 | 450.5 MiB | f8be8fc49c505731b3a95e91ebbdaed299625b6fa8a5bfb2d024b8b9282cb3d3


/kaggle/working/paper5_d2_runs/otsu_hd7zh1yp/best_validation.weights.h5

PASS D2_Stretch_Only_Epoch6.weights.h5 | 450.5 MiB | 870dada3526eaa8738db6f07349f1486cec0afa6e99a259d83704048e5bbfd71


/kaggle/working/paper5_d2_runs/stretch_only_v2odbrhu/best_validation.weights.h5

PASS D2_amff_shared_init.weights.h5 | 225.3 MiB | 545cade231e29e4c552ef469e0d0205d567fb2efa4b82e5fe720715f9a9e0506


/kaggle/working/paper5_d2_initialization/D2_amff_shared_init.weights.h5

Evidence archive: 12 files plus inventory | 6.0 MiB
D2 BACKUP READY: download the evidence ZIP and three checkpoints.


/kaggle/working/D2_Paper5_Evidence_Backup.zip